# ML Model Training and Evaluation


In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_validate
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("scotiabank_ml")

train = pd.read_csv("../data/splits/train.csv")

FEATURES = ["X1","X2","X3","X4","X6","X7","X8","X10","X11","X12",
            "X3_missing","X4_missing","X9_encoded"]
TARGET = "TARGET"

X_train = train[FEATURES]
y_train = train[TARGET]

In [12]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=20,
    min_samples_split= 50,
    max_features='sqrt',
    n_jobs=-1, random_state=42)

cv_results = cross_validate(
    rf, X_train, y_train,
    cv=5,
    scoring=["neg_root_mean_squared_error", "r2"],
    return_train_score=True
)

rmse_val   = -cv_results["test_neg_root_mean_squared_error"].mean()
rmse_std   = cv_results["test_neg_root_mean_squared_error"].std()
r2_val     = cv_results["test_r2"].mean()
r2_std     = cv_results["test_r2"].std()
rmse_train = -cv_results["train_neg_root_mean_squared_error"].mean()
r2_train   = cv_results["train_r2"].mean()

with mlflow.start_run(run_name="random_forest"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("val_rmse_mean", rmse_val)
    mlflow.log_metric("val_rmse_std", rmse_std)
    mlflow.log_metric("val_r2_mean", r2_val)
    mlflow.log_metric("val_r2_std", r2_std)
    mlflow.log_metric("train_rmse_mean", rmse_train)
    mlflow.log_metric("train_r2_mean", r2_train)

    rf.fit(X_train, y_train)
    mlflow.sklearn.log_model(rf, "rf_model")

print(f"RF | Val RMSE: {rmse_val:.4f} ± {rmse_std:.4f} | Val R²: {r2_val:.4f} ± {r2_std:.4f}")
print(f"RF | Train RMSE: {rmse_train:.4f} | Train R²: {r2_train:.4f}")

2026/06/26 02:09:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/26 02:09:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run random_forest at: http://127.0.0.1:5000/#/experiments/2/runs/2265fc97bec24536a14fb4b0977eab8c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
RF | Val RMSE: 0.5151 ± 0.0042 | Val R²: 0.3830 ± 0.0104
RF | Train RMSE: 0.4913 | Train R²: 0.4389


In [10]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

cv_results = cross_validate(
    xgb, X_train, y_train,
    cv=5,
    scoring=["neg_root_mean_squared_error", "r2"],
    return_train_score=True
)

rmse_val   = -cv_results["test_neg_root_mean_squared_error"].mean()
rmse_std   = cv_results["test_neg_root_mean_squared_error"].std()
r2_val     = cv_results["test_r2"].mean()
r2_std     = cv_results["test_r2"].std()
rmse_train = -cv_results["train_neg_root_mean_squared_error"].mean()
r2_train   = cv_results["train_r2"].mean()

with mlflow.start_run(run_name="xgboost"):
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("val_rmse_mean", rmse_val)
    mlflow.log_metric("val_rmse_std", rmse_std)
    mlflow.log_metric("val_r2_mean", r2_val)
    mlflow.log_metric("val_r2_std", r2_std)
    mlflow.log_metric("train_rmse_mean", rmse_train)
    mlflow.log_metric("train_r2_mean", r2_train)

    xgb.fit(X_train, y_train)
    mlflow.sklearn.log_model(xgb, "xgb_model")

print(f"XGB | Val RMSE: {rmse_val:.4f} ± {rmse_std:.4f} | Val R²: {r2_val:.4f} ± {r2_std:.4f}")
print(f"XGB | Train RMSE: {rmse_train:.4f} | Train R²: {r2_train:.4f}")

2026/06/26 10:47:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/26 10:47:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost at: http://127.0.0.1:5000/#/experiments/2/runs/6e68655e442e4830bf97147820ef28d7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
XGB | Val RMSE: 0.5066 ± 0.0055 | Val R²: 0.4033 ± 0.0140
XGB | Train RMSE: 0.4818 | Train R²: 0.4604


In [21]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.02,
    max_depth=4,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.1,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1
)

cv_results = cross_validate(
    lgbm, X_train, y_train,
    cv=5,
    scoring=["neg_root_mean_squared_error", "r2"],
    return_train_score=True
)

rmse_val   = -cv_results["test_neg_root_mean_squared_error"].mean()
rmse_std   = cv_results["test_neg_root_mean_squared_error"].std()
r2_val     = cv_results["test_r2"].mean()
r2_std     = cv_results["test_r2"].std()
rmse_train = -cv_results["train_neg_root_mean_squared_error"].mean()
r2_train   = cv_results["train_r2"].mean()

with mlflow.start_run(run_name="lightgbm_v1"):
    mlflow.log_param("model", "LightGBM")
    mlflow.log_param("n_estimators", 500)
    mlflow.log_param("learning_rate", 0.02)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("num_leaves", 31)
    mlflow.log_param("cv_folds", 5)


    mlflow.log_metric("val_rmse_mean", rmse_val)
    mlflow.log_metric("val_rmse_std", rmse_std)
    mlflow.log_metric("val_r2_mean", r2_val)
    mlflow.log_metric("val_r2_std", r2_std)
    mlflow.log_metric("train_rmse_mean", rmse_train)
    mlflow.log_metric("train_r2_mean", r2_train)

    lgbm.fit(X_train, y_train)
    mlflow.sklearn.log_model(lgbm, "lgbm_model")

print(f"LGBM | Val RMSE: {rmse_val:.4f} ± {rmse_std:.4f} | Val R²: {r2_val:.4f} ± {r2_std:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000840 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2063
[LightGBM] [Info] Number of data points in the train set: 26667, number of used features: 13
[LightGBM] [Info] Start training from score 8.827934
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

2026/06/26 02:35:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

2026/06/26 02:35:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run lightgbm_v1 at: http://127.0.0.1:5000/#/experiments/2/runs/64508110b8fd4c7c9d7c9b5c820e53c8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
LGBM | Val RMSE: 0.5105 ± 0.0054 | Val R²: 0.3941 ± 0.0130


In [3]:
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)

cv_results = cross_validate(
    mlp, X_train, y_train,
    cv=5,
    scoring=["neg_root_mean_squared_error", "r2"],
    return_train_score=True
)

rmse_val   = -cv_results["test_neg_root_mean_squared_error"].mean()
rmse_std   = cv_results["test_neg_root_mean_squared_error"].std()
r2_val     = cv_results["test_r2"].mean()
r2_std     = cv_results["test_r2"].std()
rmse_train = -cv_results["train_neg_root_mean_squared_error"].mean()
r2_train   = cv_results["train_r2"].mean()

with mlflow.start_run(run_name="mlp_v1"):
    mlflow.log_param("model", "MLP")
    mlflow.log_param("hidden_layer_sizes", "(128, 64, 32)")
    mlflow.log_param("activation", "relu")
    mlflow.log_param("learning_rate_init", 0.001)
    mlflow.log_param("max_iter", 300)
    mlflow.log_param("early_stopping", True)
    mlflow.log_param("cv_folds", 5)

    mlflow.log_metric("val_rmse_mean", rmse_val)
    mlflow.log_metric("val_rmse_std", rmse_std)
    mlflow.log_metric("val_r2_mean", r2_val)
    mlflow.log_metric("val_r2_std", r2_std)
    mlflow.log_metric("train_rmse_mean", rmse_train)
    mlflow.log_metric("train_r2_mean", r2_train)

    mlp.fit(X_train, y_train)
    mlflow.sklearn.log_model(mlp, "mlp_model")

print(f"MLP | Val RMSE: {rmse_val:.4f} ± {rmse_std:.4f} | Val R²: {r2_val:.4f} ± {r2_std:.4f}")
print(f"MLP | Train RMSE: {rmse_train:.4f} | Train R²: {r2_train:.4f}")

2026/06/26 10:28:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/26 10:28:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run mlp_v1 at: http://127.0.0.1:5000/#/experiments/2/runs/f6788325ea1c49cb8ba4abc85a79c6de
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
MLP | Val RMSE: 0.5234 ± 0.0029 | Val R²: 0.3630 ± 0.0105
MLP | Train RMSE: 0.5091 | Train R²: 0.3976
